# Transfer learning and CNN mechanics

A pretrained network is useful because its early layers already encode reusable visual structure. This notebook separates two ideas that are often blended together: how convolution changes a tensor, and how transfer learning changes the optimization problem.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

sns.set_theme(style='whitegrid', context='notebook')
torch.manual_seed(4)

## What a convolution sees

A kernel is a small learned array that is moved across the image. At each location it computes a weighted sum of nearby pixels. Padding controls whether border information is discarded; stride controls how far the kernel moves between outputs. The receptive field grows as layers are stacked.

In [ ]:
image = torch.zeros(1, 1, 9, 9)
image[0, 0, 2:7, 4] = 1
image[0, 0, 4, 2:7] = 1
edge_detector = torch.tensor([[-1., -1., -1.], [0., 0., 0.], [1., 1., 1.]]).view(1,1,3,3)
responses = {
    'no padding, stride 1': nn.functional.conv2d(image, edge_detector, stride=1),
    'padding 1, stride 1': nn.functional.conv2d(image, edge_detector, padding=1, stride=1),
    'padding 1, stride 2': nn.functional.conv2d(image, edge_detector, padding=1, stride=2),
}
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(image[0,0], cmap='gray'); axes[0].set_title('input 9×9')
for ax, (title, response) in zip(axes[1:], responses.items()):
    ax.imshow(response[0,0], cmap='coolwarm'); ax.set_title(f'{title}\n{tuple(response.shape[-2:])}')
for ax in axes: ax.axis('off')
plt.tight_layout()

## A residual block

A plain stack must transform its input completely. A residual block instead learns a correction and adds the original activation back:

`output = activation(F(x) + x)`

When the correction is small, the skip path gives gradients a direct route through the block. This does not make every network easy to train, but it changes the optimization geometry in a useful way.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
    def forward(self, x):
        return torch.relu(x + self.body(x))

block = ResidualBlock(8)
x = torch.randn(2, 8, 16, 16, requires_grad=True)
y = block(x); y.mean().backward()
print('input:', tuple(x.shape), 'output:', tuple(y.shape), 'gradient norm:', x.grad.norm().item())

## Freeze, replace the head, then fine-tune

Transfer learning starts with a pretrained feature extractor and a task-specific head. Freezing the backbone prevents the small target dataset from immediately changing useful generic features. After the new head learns, unfreezing with a smaller learning rate lets the representation adapt without erasing it.

In [ ]:
weights = ResNet18_Weights.DEFAULT
backbone = resnet18(weights=weights)
for parameter in backbone.parameters():
    parameter.requires_grad = False
input_features = backbone.fc.in_features
backbone.fc = nn.Linear(input_features, 2)
trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
print(f'trainable parameters after replacing the head: {trainable:,} / {total:,}')

# The two optimization stages are explicit even when a high-level library manages them.
head_optimizer = torch.optim.Adam(backbone.fc.parameters(), lr=1e-3)
for parameter in backbone.layer4.parameters():
    parameter.requires_grad = True
finetune_optimizer = torch.optim.Adam([
    {'params': backbone.layer4.parameters(), 'lr': 1e-4},
    {'params': backbone.fc.parameters(), 'lr': 1e-3},
])
print('two parameter groups:', [group['lr'] for group in finetune_optimizer.param_groups])

The pretrained weights download only when this cell is run with an internet connection and a populated torchvision cache. The mechanics above are still inspectable offline. In a project notebook, record the exact weights, preprocessing transform, frozen layers, and learning-rate schedule; those choices are part of the experiment.